# rnaSPAdes — сборка BCR из симулированных PE150 — человек PRJEB30386

Один смешанный образец IGH+IGK+IGL (4 библиотеки 5′MTPX).

rnaSPAdes (SPAdes 4.3, режим RNA) — универсальный de novo сборщик транскриптома по графу де Брёйна, без знаний о V(D)J. Для PE150 размеры k-мер подбираются автоматически. Контиги — `transcripts.fasta`.

Выход для каждого образца: `<ветка>/assemblers/rnaspades/<образец>/` — нативные файлы сборщика, `run.log` и нормализованный `contigs.fasta` для сравнения с эталоном (`benchmark_assemblers_human.ipynb`). Готовый результат не пересчитывается (`FORCE=False`).

## Окружение и параметры

In [ ]:
import os, sys, shutil, subprocess, time, gzip, json
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]

DATASET = "PRJEB30386"
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_umicons_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["PRJEB30386_all_chains"]
THREADS = int(os.environ.get("BCR_THREADS", 20))
FORCE = False

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
DATASET_DIR = ROOT / "results" / DATASET
BRANCH_DIR = DATASET_DIR / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
ASM_DIR = BRANCH_DIR / "assemblers"

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

def iter_fasta(path):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    """Нормализованный FASTA контигов для сравнения с эталоном."""
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

print("ROOT:", ROOT); print("BRANCH:", BRANCH_DIR); print("samples:", SAMPLES, "threads:", THREADS)


## Запуск rnaSPAdes

In [ ]:
MEMORY_GB = int(os.environ.get("BCR_MEMORY_GB", 40))   # лимит памяти задачи OneQ — 64 ГБ
print(subprocess.run(["rnaspades.py", "--version"], capture_output=True, text=True).stdout.strip())

for sample in SAMPLES:
    out = ASM_DIR / "rnaspades" / sample
    contigs = out / "contigs.fasta"
    if contigs.exists() and not FORCE:
        print("[skip]", sample); continue
    r1, r2 = reads(sample)
    work = out / "spades"
    if work.exists():
        shutil.rmtree(work)
    out.mkdir(parents=True, exist_ok=True)
    run_logged(["rnaspades.py", "-1", r1, "-2", r2, "-o", work, "-t", THREADS, "-m", MEMORY_GB], out / "run.log")
    # Если длину вставки оценить не удалось (граф из множества близких клонов), rnaSPAdes пропускает
    # разрешение повторов и не пишет transcripts.fasta; тогда берутся финальные контиги последнего k.
    fasta = work / "transcripts.fasta"
    if not fasta.exists():
        fasta = sorted(work.glob("K*/final_contigs.fasta"), key=lambda p: int(p.parent.name[1:]))[-1]
    print("rnaSPAdes output used:", fasta)
    (out / "contigs_source.txt").write_text(str(fasta) + "\n")
    write_contigs(iter_fasta(fasta), contigs, "rnaspades")
